### RAG Pipelines - Data Ingestion to Vector DB pipeline

In [6]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [10]:
from langchain_core import documents
## data ingestion - reading all the pdfs

def process_all_pdfs(pdf_directory):
    """Process all the PDF files in directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all Pdf files recursively

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} pdf files to process")

    for pdf_file in pdf_files:
        print(f"\nProcesssing : {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source info to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"✓ Loaded {len(documents)} pages")
        
        except Exception as e:
            print("✗ Error:{e}")

    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 3 pdf files to process

Processsing : ragg.pdf
✓ Loaded 7 pages

Processsing : rag_arch.pdf
✓ Loaded 8 pages

Processsing : rag_article.pdf
✓ Loaded 19 pages

 Total documents loaded: 34


In [11]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m120', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) obsidian/1.5.12 Chrome/120.0.6099.283 Electron/28.2.3 Safari/537.36', 'creationdate': '2024-09-29T07:50:14+00:00', 'moddate': '2024-09-29T07:50:14+00:00', 'source': '..\\data\\pdf\\ragg.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'ragg.pdf', 'file_type': 'pdf'}, page_content='Retrieval Augmented Generation\n"RAG takes input and retrieves a set of relevant/supporting documents given a source (e.g.,\nWikipedia). The documents are concatenated as context with the original input prompt and fed\nto the text generator which produces the final output. This makes RAG adaptive for situations\nwhere facts could evolve over time. This is very useful as Large Language Models\'s parametric\nknowledge is static. RAG allows language models to bypass retraining, enabling access to the\nlatest information for generating reliable outputs v

In [12]:
from sqlalchemy import text
from langchain_core.messages.utils import _default_text_splitter
## Text splitting - into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ","",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content {split_docs[0].page_content[:200]}")
        print(f"Metadata {split_docs[0].metadata}")

    return split_docs

In [14]:
chunks = split_documents(all_pdf_documents)
chunks

Split 34 documents into 59 chunks

Example chunk:
Content Retrieval Augmented Generation
"RAG takes input and retrieves a set of relevant/supporting documents given a source (e.g.,
Wikipedia). The documents are concatenated as context with the original input
Metadata {'producer': 'Skia/PDF m120', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) obsidian/1.5.12 Chrome/120.0.6099.283 Electron/28.2.3 Safari/537.36', 'creationdate': '2024-09-29T07:50:14+00:00', 'moddate': '2024-09-29T07:50:14+00:00', 'source': '..\\data\\pdf\\ragg.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'ragg.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m120', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) obsidian/1.5.12 Chrome/120.0.6099.283 Electron/28.2.3 Safari/537.36', 'creationdate': '2024-09-29T07:50:14+00:00', 'moddate': '2024-09-29T07:50:14+00:00', 'source': '..\\data\\pdf\\ragg.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'ragg.pdf', 'file_type': 'pdf'}, page_content='Retrieval Augmented Generation\n"RAG takes input and retrieves a set of relevant/supporting documents given a source (e.g.,\nWikipedia). The documents are concatenated as context with the original input prompt and fed\nto the text generator which produces the final output. This makes RAG adaptive for situations\nwhere facts could evolve over time. This is very useful as Large Language Models\'s parametric\nknowledge is static. RAG allows language models to bypass retraining, enabling access to the\nlatest information for generating reliable outputs v

## Embedding and VectorStore

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SenctenceTrasformer"""

    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name : HugginFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None # we will set this later
        self._load_model() #_ underscore because this is a protected class and can be used under this function only

    def _load_model(self):
        """Load the SenctenceTransformer model"""

        try:
            print(f"Loading embedding model{self.model_name}")
            self.model_name = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise

    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args: 
            texts: List of text strings to embed

        Returns:
        numpy array of embeddings with shape (len(texts), embedding_dim)

        """

        if not self.model:
            raise ValueError("model not loaded")

        print(f"Generatig embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    # just to get the embedding dimension 
    # def get_embedding_dimension(self):
    #     """Get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #         return self.model.get_sentence_embedding_dimension()


## we can initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding modelall-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7923.36it/s]


Error loading model SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
):'NoneType' object has no attribute 'get_sentence_embedding_dimension'


AttributeError: 'NoneType' object has no attribute 'get_sentence_embedding_dimension'